In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install insightface onnxruntime-gpu mediapipe opencv-python scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 8.6 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 7.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.3 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071486 sha256=301708b2bb07f00971c8636a3ae356c6c07745b4638ffaba23e951c307c44a59
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not curre

In [2]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis

In [3]:
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']

app = FaceAnalysis(name="buffalo_l", providers=providers)
app.prepare(ctx_id=0, det_size=(640,640))

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 96753.13KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

In [4]:
def get_embedding_from_image(image):
    try:
        faces = app.get(image)

        if len(faces) == 0:
            return None

        # choose largest face
        faces = sorted(
            faces,
            key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]),
            reverse=True
        )

        emb = faces[0].embedding
        emb = emb / np.linalg.norm(emb)

        return emb

    except Exception as e:
        return None

In [6]:
reference_path = "//kaggle/input/datasets/siyabehera/videotest/refernce.jpeg"

ref_img = cv2.imread(reference_path)

ref_emb = get_embedding_from_image(ref_img)

if ref_emb is None:
    print("No face detected in reference image")
else:
    print("Reference embedding created")

Reference embedding created


In [7]:
def verify_video(video_path, ref_emb, fps_process=5):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video")
        return

    video_fps = cap.get(cv2.CAP_PROP_FPS)

    if video_fps == 0:
        video_fps = 25

    frame_interval = max(1, int(video_fps / fps_process))

    similarities = []
    frame_count = 0
    processed_frames = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_count % frame_interval == 0:

            emb = get_embedding_from_image(frame)

            if emb is not None:
                similarity = np.dot(ref_emb, emb)
                similarities.append(similarity)
                processed_frames += 1

        frame_count += 1

    cap.release()

    if len(similarities) == 0:
        return {
            "match": False,
            "reason": "No face detected in video"
        }

    max_similarity = float(np.max(similarities) * 100)
    avg_similarity = float(np.mean(similarities) * 100)

    threshold = 55
    match = max_similarity >= threshold

    return {
        "match": match,
        "max_similarity": max_similarity,
        "avg_similarity": avg_similarity,
        "frames_used": processed_frames
    }

In [11]:
videos = [
    "/kaggle/input/datasets/siyabehera/videotest/test.mp4.mp4",
    "/kaggle/input/datasets/siyabehera/newtestfiles/test1.mp4",
    "/kaggle/input/datasets/siyabehera/newtestfiles/test2.mp4"
]

for vid in videos:
    print("\nTesting:", vid)
    result = verify_video(vid, ref_emb)
    print(result)


Testing: /kaggle/input/datasets/siyabehera/videotest/test.mp4.mp4
{'match': True, 'max_similarity': 86.57376098632812, 'avg_similarity': 58.68694305419922, 'frames_used': 12}

Testing: /kaggle/input/datasets/siyabehera/newtestfiles/test1.mp4
{'match': True, 'max_similarity': 76.99066162109375, 'avg_similarity': 69.28258514404297, 'frames_used': 7}

Testing: /kaggle/input/datasets/siyabehera/newtestfiles/test2.mp4
{'match': True, 'max_similarity': 69.15441131591797, 'avg_similarity': 63.02909851074219, 'frames_used': 14}
